# Phase 5: The User Interface (Gradio GUI)

In the previous four phases, I successfully built, tested, and evaluated my local Zero Trust RAG architecture. However, right now, it only works via Python code in a backend terminal.

For a system to be truly useful in the real world, non-technical users (like managers or junior analysts) need an easy way to interact with it. In this final phase, I am wrapping my entire pipeline in an interactive Web GUI using the **Gradio** framework.

### What to expect in this notebook:
* **Step 1: Waking Up the Backend Architecture** – Connecting to Google Drive, securely authenticating my Hugging Face token, and loading the Knowledge Graph, Search Engine, and AI Generator into memory.
* **Step 2: Building and Launching the Web Application** – Designing the frontend layout, writing the core logic to connect the chat UI to my backend Python pipeline, and launching the live interactive server.

### Step 1: Waking Up the Backend Architecture

Before I can build the frontend, I need to bring the backend online one last time.

In this cell, I am:
1. **Connecting Storage:** Mounting Google Drive so Python can find my custom scripts and databases.
2. **Authenticating Safely:** Logging into Hugging Face to access the Llama-3 model. *(Notice that I am now securely using Colab Secrets `userdata.get()` instead of hardcoding my token!)*
3. **Initializing the Pipeline:** I am importing the custom `AdvancedRetriever` and `AdvancedGenerator` modules I wrote in Phases 2 and 3, and loading them into the GPU.

Once this cell finishes running, my entire Zero Trust Architect is online and ready to be plugged into a user interface.

In [1]:
# Install core pipeline tools and Gradio for the UI
!pip install -q transformers accelerate bitsandbytes langchain langchain-community langchain-huggingface networkx huggingface_hub sentence-transformers chromadb gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 128.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 124.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7

In [2]:
import os
import sys
import pickle
from google.colab import drive, userdata
from huggingface_hub import login

# 1. Mount Drive & Add Path
drive.mount('/content/drive')
project_path = '/content/drive/My Drive/ZTA_Project'
sys.path.append(project_path)

# 2. Force Authentication
MY_TOKEN = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = MY_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = MY_TOKEN
login(token=MY_TOKEN, add_to_git_credential=True)

# 3. Initialize the Models
from retrieval import AdvancedRetriever
from generator import AdvancedGenerator

# Load Graph
with open(os.path.join(project_path, 'zta_knowledge_graph.gpickle'), 'rb') as f:
    G = pickle.load(f)

# Load Search Engine
retriever = AdvancedRetriever('/content/drive/My Drive/ZTA_Project/chroma_db_bge')

# Load LLM
generator = AdvancedGenerator(G)

print("Developer Log: System Architecture Online. Ready for UI integration.")

Mounted at /content/drive


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Initializing Advanced Retriever on CUDA...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Developer Log: System Architecture Online. Ready for UI integration.


### Step 2: Building and Launching the Gradio Web Application

Now for the exciting part. In this cell, I am writing the code to generate the actual visual interface and connect it directly to my backend AI pipeline.

Here is exactly how I built this web application:
1. **The Core Logic (The Bridge):** I wrote a function that takes whatever the user types into the chat box, feeds it through my Phase 2 Search Engine, and passes those documents to my Phase 3 Llama-3 model to generate a response.
2. **Dynamic Citations:** To enforce Zero Trust transparency, I added logic that extracts the exact PDF page numbers and raw text snippets from the database metadata. If the user wants to verify a claim, the AI will print its exact sources!
3. **The Front-End Layout:** I designed a clean, professional dashboard using Gradio Blocks. It features a large central chat window and a right-hand settings panel where users can actively toggle the citations on or off.
4. **Going Live:** Finally, I launch the application. By setting `share=True`, Gradio generates a secure, temporary public web link so anyone can test my AI Architect directly from their browser!

In [4]:
import gradio as gr

# ==========================================
# 1. THE CORE LOGIC FUNCTION
# ==========================================
def chat_with_architect(user_message, chat_history, show_citations):
    """This function connects the web UI to your RAG pipeline."""

    # Step 1: Retrieve and Rerank
    retrieval_results = retriever.retrieve_and_rerank(user_message)
    docs = retrieval_results['documents']

    # Step 2: Generate the Answer
    answer = generator.generate_answer(user_message, docs)

    # Step 3: Append Citations if the user flipped the toggle
    if show_citations:
        answer += "\n\n### 📚 Sources & Citations:\n"
        for i, doc in enumerate(docs, 1):
            # Extract a snippet of the raw text
            snippet = doc.page_content.replace('\n', ' ')[:150] + "..."

            # Grab the document name
            source_name = doc.metadata.get('source', 'NIST SP 800-207 Document')

            # Grab the specific page number from the metadata
            page_num = doc.metadata.get('page')

            # Format the label nicely. (LangChain PDF loaders are often 0-indexed, so we add 1)
            if page_num is not None:
                source_display = f"{source_name} (Page {int(page_num) + 1})"
            else:
                source_display = source_name

            answer += f"**[{i}] {source_display}**\n> *\"{snippet}\"*\n\n"

    # Step 4: Update the chat history
    chat_history.append((user_message, answer))
    return "", chat_history


# ==========================================
# 2. THE FRONT-END DESIGN
# ==========================================
# We use Gradio Blocks to create a custom, professional layout
with gr.Blocks(theme=gr.themes.Soft()) as zta_app:

    # Header
    gr.Markdown("# 🛡️ Zero Trust Architecture (ZTA) AI Advisor - Student: Nithin, ID: a1943553, Assignment 3: Retrieval-Augmented Generation (RAG) System")
    gr.Markdown("Ask technical questions regarding NIST 800-207 policy deployment. Powered by an Advanced RAG pipeline, local Llama-3-8B inference, and knowledge graph extraction.")

    with gr.Row():

        # Left Column: The Chat Interface
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(height=550, label="Senior Security Architect")
            user_input = gr.Textbox(placeholder="E.g., How do I secure a compromised remote worker's laptop?", label="Your Question")

            with gr.Row():
                submit_btn = gr.Button("Submit", variant="primary")
                clear_btn = gr.ClearButton([user_input, chatbot], value="Clear Conversation")

        # Right Column: Settings and Transparency
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ System Settings")

            # THE CITATION TOGGLE
            citation_toggle = gr.Checkbox(label="Show NIST Citations", value=True, info="Append the retrieved architectural chunks to the response to verify accuracy.")

            gr.Markdown("---")
            gr.Markdown("### 📊 System Status\n- ✅ Vector Database (ChromaDB)\n- ✅ Cross-Encoder Reranking\n- ✅ Context Compression\n- ✅ Llama-3 (4-bit Quantized)")

    # ==========================================
    # 3. EVENT LISTENERS (Triggers)
    # ==========================================
    # Pressing Enter in the textbox or clicking Submit triggers the logic
    user_input.submit(chat_with_architect, inputs=[user_input, chatbot, citation_toggle], outputs=[user_input, chatbot])
    submit_btn.click(chat_with_architect, inputs=[user_input, chatbot, citation_toggle], outputs=[user_input, chatbot])

# ==========================================
# 4. LAUNCH THE SERVER
# ==========================================
# share=True creates a public URL you can put in your project report!
zta_app.launch(share=True, debug=True)

/tmp/ipykernel_1708/349211155.py:46: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as zta_app:
/tmp/ipykernel_1708/349211155.py:56: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=550, label="Senior Security Architect")
/tmp/ipykernel_1708/349211155.py:56: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=550, label="Senior Security Architect")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://09ef3ced63056150e9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://09ef3ced63056150e9.gradio.live


## Project Wrap-Up and Final Conclusions

Building this 5-phase modular architecture successfully demonstrates how a localized, highly secure RAG system can be deployed for sensitive cybersecurity environments.

By applying the advanced techniques covered in our lecture slides—specifically **Pre-Retrieval query expansion** and **Post-Retrieval cross-encoder reranking**—I was able to solve the common issues of AI hallucination and poor search relevance. Furthermore, utilizing a **Hybrid Data Source** (combining Vector Embeddings with a Knowledge Graph) ensured the AI understood the structural relationships between complex Zero Trust concepts, not just the raw text.

**The Final Result:**
The pipeline successfully digested massive, dense government frameworks and translated them into a fast, highly accurate, and fully interactive Web GUI (built in Phase 5). Most importantly, by keeping the entire Llama-3 generation and RAGAS evaluation loop 100% local on the GPU, this architecture strictly adhered to the very Zero Trust privacy principles it was designed to teach.